In [ ]:
from pyspark.errors import PySparkException
from py4j.protocol import Py4JJavaError
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F

In [ ]:
# Recupera as credenciais de acesso ao banco de dados utilizando o Databricks Secrets
user = dbutils.secrets.get("databases_lh_nautical", "sql_server_user")
password = dbutils.secrets.get("databases_lh_nautical", "sql_server_password")
host = dbutils.secrets.get("databases_lh_nautical", "sql_server_host")
port = dbutils.secrets.get("databases_lh_nautical", "sql_server_port")

In [ ]:
def query_jdbc(jdbc_url, conn, full_table_name, num_partitions, parallel_reference_column = None):
    """Realiza a leitura de uma tabela via JDBC, permitindo a paralelização da carga.

    Args:
        jdbc_url (str): URL de conexão com o banco de dados.
        conn (dict): Dicionário com credenciais e driver de conexão.
        full_table_name (str): Nome da tabela de origem (ex: 'dbo.clientes' ou subquery).
        num_partitions (int/str): Quantidade de partições que o Spark usará para ler os dados.
        parallel_reference_column (str, optional): Coluna numérica (id) ou de data usada para
            dividir a carga entre as partições. Padrão é None (leitura sequencial).

    Returns:
        DataFrame: Spark DataFrame contendo os dados resultantes da consulta.
    """
    # Se uma coluna de partição for informada, calcula os limites (MIN/MAX) para paralelizar a leitura
    if parallel_reference_column is not None:
        limits_df = spark.read.jdbc(
            url = jdbc_url,
            table = f"(SELECT MIN({parallel_reference_column}) AS min_atributo, MAX({parallel_reference_column}) AS max_atributo FROM {full_table_name}) AS bounds",
            properties = conn
        )

        limits = limits_df.collect()[0]
        lower_bound = limits['min_atributo']
        upper_bound = limits['max_atributo']

        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = full_table_name,
            column = parallel_reference_column, # Coluna usada para paralelização
            lowerBound = lower_bound if lower_bound is not None else None, # Limite inferior
            upperBound = upper_bound if upper_bound is not None else None, # Limite superior 
            numPartitions = int(num_partitions), # Número de partições usada na paralelização
            properties = conn
        )
    else:
        df_temp = spark.read.jdbc(
            url = jdbc_url, 
            table = full_table_name,
            properties = conn
        )
    
    return df_temp

In [ ]:
def incremental_ingenstion_hash(jdbc_url, conn, table_name, catalog_name, num_partition, parallel_reference_column = None):
    """Gerencia o pipeline de ingestão de dados combinando JDBC e a estratégia de Hashing (MD5).

    Esta função automatiza a captura de alterações (CDC lógico) na camada de staging. 
    Ela calcula dinamicamente uma "impressão digital" (_hash) para cada linha da tabela de 
    origem. Em cargas subsequentes, em vez de comparar a tabela inteira coluna por coluna, o 
    Spark realiza um join otimizado (left_anti) trazendo apenas a coluna de hash da última versão 
    do Delta Lake (Time Travel), isolando de forma performática registros novos e modificados.

    Args:
        jdbc_url (str): URL de conexão JDBC formatada para o banco de dados de origem.
        conn (dict): Dicionário contendo as credenciais de autenticação e especificações do driver.
        table_name (str): Nome simples da tabela no banco de dados relacional (ex: 'clientes').
        catalog_name (str): Caminho do catálogo e schema de destino no Unity Catalog.
        num_partition (int or str): Quantidade de partições que o Spark usará para fatiar e paralelizar a leitura dos dados via JDBC.
        parallel_reference_column (str, optional): Nome da coluna numérica (ex: 'id') ou de data 
            (ex: 'updated_at') utilizada para calcular os limites (lower/upper bounds) da query JDBC. 
            Se omitido (None), a leitura será estritamente sequencial (uma única thread).

    Returns:
        tuple: Uma tupla contendo dois elementos (DataFrame, str) ou (None, None):
            - elemento 0 (DataFrame or None): Spark DataFrame contendo apenas as linhas novas ou 
              alteradas, já acrescido das colunas de auditoria (`_ingested_at`, `_action`, `_hash`). 
              Retorna `None` se nenhuma alteração for detectada.
            - elemento 1 (str or None): Sugestão do modo de gravação para o Delta Lake 
              (`"overwrite"` para cargas iniciais/tabelas legadas, `"append"` para deltas incrementais, 
              ou `None` se não houver alterações).
    """
    source_table_name = f"dbo.{table_name}"
    destination_table_name = f"{catalog_name}.{table_name}"

    # 1. Busca os dados atuais da origem via JDBC
    df_temp = query_jdbc(jdbc_url, conn, source_table_name, num_partition, parallel_reference_column)
    
    # 2. Cria a coluna de Hash dinamicamente combinando TODAS as colunas originais da tabela.
    # Tratamos valores nulos com coalesce para garantir que IDs nulos ou strings vazias não colidam hashes.
    original_columns = df_temp.columns
    df_source_with_hash = df_temp.withColumn(
        "_hash", F.md5(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in original_columns]))
    )

    # Quantidade de linhas na tabela de origem
    qty_source_rows = df_temp.count()
    print(f"[{table_name}] A quantidade de linhas na tabela origem (SLQ Server): {qty_source_rows}")
    

    # Cenário 1: Tabela não existe no destino -> Carga Inicial
    if not spark.catalog.tableExists(destination_table_name):
        print(f"[{table_name}] Primeira ingestão (Carga Inicial com Hash) da tabela: {table_name}")
        df_temp_with_action_date = df_source_with_hash \
            .withColumn("_ingested_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
            .withColumn("_action", F.lit("INITIAL_LOAD"))
        
        return df_temp_with_action_date, "overwrite"
            
    # Cenário 2: Tabela já existe -> Filtra deltas usando comparação rápida de Hash
    else:
        # Pega as colunas atuais da tabela destino antes de ler
        destination_columns = spark.table(destination_table_name).columns
        
        # Se a tabela existe mas NÃO tem a coluna '_hash' (legado), força o comportamento de Carga Inicial
        if "_hash" not in destination_columns:
            print(f"[{table_name}] A tabela {table_name} existe, mas não possui a coluna '_hash'. Forçando carga inicial para atualizar schema...")
            df_temp_with_action_date = df_source_with_hash \
                .withColumn("_ingested_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
                .withColumn("_action", F.lit("INITIAL_LOAD"))
            
            print(f"[{table_name}] Primeira ingestão (Carga Inicial com Hash) da tabela: {table_name}")
            return df_temp_with_action_date, "overwrite"
            
        # Obtém o histórico mais recente do Delta Lake para evitar leituras de snapshots fantasmas
        latest_version = spark.sql(f"DESCRIBE HISTORY {destination_table_name}").first()["version"]
        
        # Lê o destino trazendo APENAS a coluna de hash para a memória (I/O mínimo)
        print(f"[{table_name}] Verificando se há alterações entre as tabelas")
        df_destination_hashes = spark.read.format("delta") \
            .option("versionAsOf", latest_version) \
            .table(destination_table_name) \
            .select("_hash")
                
        # O left_anti join retorna tudo o que está na origem mas NÃO foi encontrado no destino
        df_deltas = df_source_with_hash.join(df_destination_hashes, on="_hash", how="left_anti")
        num_changes = df_deltas.count()
        
        if num_changes > 0:
            df_temp_with_action_date = df_deltas \
                .withColumn("_ingested_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
                .withColumn("_action", F.lit("INSERT_OR_UPDATE"))
            
            print(f"[{table_name}] Foram identificados {num_changes} registros novos/alterados na tabela {table_name}")
            return df_temp_with_action_date, "append"
        else:
            print(f"[{table_name}] Nenhuma alteração detectada para a tabela {table_name}.")
            return None, None

In [ ]:
# Montagem da String de Conexão JDBC para SQL Server
jdbc_url = f"jdbc:sqlserver://{host}:{port};databaseName=lh_nautical;encrypt=false;trustServerCertificate=true;"
conn = {
    "user": user,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Coleta de parâmetros operacionais definidos no bundle
catalog_name = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")
num_partitions = dbutils.widgets.get("numPartitions")

In [ ]:
# Consulta dinâmica no Information Schema para identificar todas as Tabelas Base do schema 'dbo'
query_tables = "(SELECT table_name FROM information_schema.tables WHERE table_schema = 'dbo' AND table_type = 'Base table') AS tabelas"

try:
    # Executa a busca e extrai os nomes das tabelas para uma lista Python nativa
    df_tables = query_jdbc(jdbc_url, conn, query_tables, num_partitions)
    list_tables = [row['table_name'] for row in df_tables.collect()]

except Py4JJavaError as e:
    msg_error = str(e)
    print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados.")
    print(f"Detalhes do erro: {msg_error}")
    list_tables = []
    raise e
  
except PySparkException as e:
    msg_error = str(e)
    print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {msg_error}")
    list_tables = []
    raise e

except Exception as e:
    print(f"[Erro Inesperado] - {str(e)}")
    list_tables = []
    raise e

In [ ]:
# Ingestão de dados
tables_with_errors = []
full_catalog_name = f"{catalog_name}.{schema}"

# Iteração sobre a lista de tabelas para processamento e salvamento
for table_name in list_tables:
    print(f"\nImportanto a tabela: {table_name}...")
    destination_table_name = f"{full_catalog_name}.{table_name}"
    
    try:
        # Regra de Negócio: Define a coluna de paralelização com base nas características da tabela
        if table_name not in ['variant_attribute_values', 'product_suppliers', 'stock_levels']:
            # Tabelas padrão utilizam a chave primária incremental 'id'
            df_ingestion, write_mode = incremental_ingenstion_hash(jdbc_url, conn, table_name, full_catalog_name, num_partitions, "id")

        elif table_name in ['product_suppliers', 'stock_levels']:
            # Tabelas de transição utilizam carimbo de data 'updated_at'
            df_ingestion, write_mode = incremental_ingenstion_hash(jdbc_url, conn, table_name, full_catalog_name, num_partitions, "updated_at")

        else:
            # Caso caia em alguma exceção não mapeada, executa sem chave de partição (sequencial)
            df_ingestion, write_mode = incremental_ingenstion_hash(jdbc_url, conn, table_name, full_catalog_name, num_partitions)

        # Ingestão das tabelas com proteção dinâmica contra duplicações
        if df_ingestion is not None:
            # aplica 'overwrite' para não duplicar dados. Caso contrário, faz 'append'.
            if write_mode == "overwrite" and spark.catalog.tableExists(destination_table_name):
                print(f"[{table_name}] Tabela legada detectada. Aplicando OVERWRITE para reformatação segura de schema.")


            df_ingestion.write.format("delta") \
                .mode(write_mode) \
                .option("mergeSchema", "true") \
                .saveAsTable(destination_table_name)
            
        qty_destination_rows = spark.table(f"{destination_table_name}").count()
        print(f"[{table_name}] Quantidade de Linhas no Delta: {qty_destination_rows}")
            
    except Py4JJavaError as e:
        error_msg = str(e)
        print("[Erro de conexão/JDBC] - Falha ao conectar ou executar a query no banco de dados." )
        print(f"Erro ao processar tabela {table_name}: {error_msg}")
        # Registra a tabela com falha no relatório final e propaga o erro para alertar a orquestração
        tables_with_errors.append(table_name)
        continue
    
    except PySparkException as e:
        error_msg = str(e)
        print(f"[Erro do Spark] - Ocorreu uma falha interna no Spark: {error_msg}")
        print(f"Erro ao processar tabela {table_name}: {error_msg}")
        # Registra a tabela que falhou para o resumo final
        tables_with_errors.append(table_name)
        # Dispara o e-mail de alerta
        continue
    
    except Exception as e:
        error_msg = str(e)
        print(f"[Erro Inesperado] - {error_msg}")
        # Registra a tabela que falhou para o resumo final
        tables_with_errors.append(table_name)
        # Dispara o e-mail de alerta
        continue

# Resumo da execução do Pipeline
print("\nIngestão finalizada!")
if tables_with_errors:
    print(f"Atenção! As seguintes tabelas falharam e geraram alertas: {tables_with_errors}")
    raise Exception(f"Falha na ingestão das seguintes tabelas: {tables_with_errors}")
else:
    print("Sucesso absoluto! Todas as tabelas foram copiadas sem erros.")